# EoMT + Qwen3 FP8 no Kaggle

Notebook pronto para importar no [Kaggle](https://www.kaggle.com/code). Contém:

1. Clone do repositório **tue-mps/eomt** (Encoder-only Mask Transformer).
2. Exemplos de como editar arquivos do repositório com IA (ler/escrever via Python).
3. Carregamento e inferência do modelo **Qwen3-8-27B-FP8** de `/kaggle/input/...`.
4. Treinamento e validação do EoMT adaptado para **1 GPU** do Kaggle.

> **Antes de rodar:** em **Settings → Accelerator**, selecione **GPU**.

In [ ]:
# ============================================================
# 1. CLONAR REPOSITÓRIO EOMT (tue-mps/eomt)
# ============================================================
import os
import subprocess
import sys

REPO_URL = "https://github.com/tue-mps/eomt.git"
DEST_DIR = "/kaggle/working/eomt"

if not os.path.exists(DEST_DIR):
    print(f"Clonando {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, DEST_DIR], check=True)
else:
    print(f"{DEST_DIR} já existe. Atualizando...")
    subprocess.run(["git", "-C", DEST_DIR, "pull"], check=True)

os.chdir(DEST_DIR)
if DEST_DIR not in sys.path:
    sys.path.insert(0, DEST_DIR)

print("Diretório de trabalho:", os.getcwd())
print("Arquivos:", os.listdir(".")[:30])

In [ ]:
# ============================================================
# 2. INSTALAR DEPENDÊNCIAS
# ============================================================
!pip install -r /kaggle/working/eomt/requirements.txt -q
!pip install -q transformers accelerate sentencepiece protobuf torch>=2.4

In [ ]:
# ============================================================
# 3. VERIFICAR GPU
# ============================================================
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("AVISO: Nenhuma GPU detectada. Ative GPU em Settings → Accelerator.")

# ============================================================
# 4. COMO EDITAR ARQUIVOS DO REPOSITÓRIO COM IA
# ============================================================

A IA não digita no notebook por você. Você copia o código sugerido por ela e cola em uma célula.

Abaixo exemplos práticos de **ler** e **escrever** arquivos do repositório clonado em `/kaggle/working/eomt`.

In [ ]:
# ============================================================
# 4a. LER UM ARQUIVO DO REPOSITÓRIO
# ============================================================

caminho = "/kaggle/working/eomt/configs/dinov2/coco/panoptic/eomt_large_640.yaml"

if os.path.exists(caminho):
    with open(caminho, "r", encoding="utf-8") as f:
        conteudo = f.read()
    print(conteudo)
else:
    print(f"Arquivo não encontrado: {caminho}")

In [ ]:
# ============================================================
# 4b. EDITAR / SOBRESCREVER UM ARQUIVO (exemplo sugerido por IA)
# ============================================================

caminho = "/kaggle/working/eomt/configs/dinov2/coco/panoptic/eomt_large_640.yaml"

novo_conteudo = """seed: 42
device: cuda
epochs: 5
batch_size: 2
learning_rate: 2e-5
output_dir: /kaggle/working/eomt/outputs
data:
  path: /kaggle/input/SEU_DATASET
"""

with open(caminho, "w", encoding="utf-8") as f:
    f.write(novo_conteudo)

print(f"Arquivo atualizado: {caminho}")

# Verificar
with open(caminho, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# ============================================================
# 4c. SUBSTITUIR TRECHO ESPECÍFICO DE UM ARQUIVO
# ============================================================

caminho = "/kaggle/working/eomt/train.py"

if os.path.exists(caminho):
    with open(caminho, "r", encoding="utf-8") as f:
        texto = f.read()
    
    texto = texto.replace(
        "batch_size: 4",
        "batch_size: 2  # ajustado para 1 GPU do Kaggle"
    )
    
    with open(caminho, "w", encoding="utf-8") as f:
        f.write(texto)
    
    print("Edição aplicada com sucesso.")
else:
    print(f"Arquivo não encontrado: {caminho}")

# ============================================================
# 5. CARREGAR MODELO QWEN3 FP8 DO /KAGGLE/INPUT
# ============================================================

Caminho do modelo no seu Kaggle:
`/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1`

Atenção: não use `device_map="auto"` com pesos FP8 já quantizados, pois pode gerar `AttributeError` no `transformers`.
Carregue na CPU primeiro, depois mova para a GPU manualmente.

In [ ]:
# ============================================================
# 5a. CARREGAR TOKENIZER E MODELO QWEN3
# ============================================================

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Modelo: carrega na CPU primeiro para evitar erro de tp_plan com FP8 + device_map auto
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,   # use torch.float16 se sua GPU não suportar bfloat16
    low_cpu_mem_usage=True,
)

# Move para GPU se disponível
if torch.cuda.is_available():
    model = model.to("cuda")
    print("Modelo carregado na GPU")
else:
    print("Modelo carregado na CPU")

print(f"Dtype: {model.dtype}")
print("Modelo Qwen3 carregado!")

In [ ]:
# ============================================================
# 5b. INFERÊNCIA (GERAR TEXTO)
# ============================================================

prompt = "Explique o que é aprendizado de máquina em duas frases."
inputs = tokenizer(prompt, return_tensors="pt")

if torch.cuda.is_available():
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(resposta)

# ============================================================
# 6. TREINAMENTO E VALIDAÇÃO EOMT (1 GPU)
# ============================================================

O EoMT usa PyTorch Lightning. No Kaggle com 1 GPU, reduza `--trainer.devices` para `1` e `--data.batch_size` para `2` (ou `1` se faltar VRAM).

Os checkpoints são salvos em `/kaggle/working/eomt/outputs`.

In [ ]:
# ============================================================
# 6a. TREINAR EOMT
# ============================================================

import subprocess

cmd = [
    "python3", "main.py", "fit",
    "-c", "configs/dinov2/coco/panoptic/eomt_large_640.yaml",
    "--trainer.devices", "1",
    "--trainer.accelerator", "gpu",
    "--data.batch_size", "2",
    "--data.path", "/kaggle/input/SEU_DATASET",  # altere para o caminho do seu dataset
    "--trainer.default_root_dir", "/kaggle/working/eomt/outputs",
    "--trainer.max_epochs", "5",
]

print("Executando:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# ============================================================
# 6b. VALIDAR / TESTAR EOMT
# ============================================================

cmd = [
    "python3", "main.py", "validate",
    "-c", "configs/dinov2/coco/panoptic/eomt_large_640.yaml",
    "--trainer.devices", "1",
    "--trainer.accelerator", "gpu",
    "--data.batch_size", "2",
    "--data.path", "/kaggle/input/SEU_DATASET",
    "--model.network.masked_attn_enabled", "False",
    "--model.ckpt_path", "/kaggle/working/eomt/outputs/last.ckpt",
]

print("Executando validação:", " ".join(cmd))
subprocess.run(cmd, check=True)

# ============================================================
# 7. CAMINHOS IMPORTANTES NO KAGGLE
# ============================================================

| Destino | Caminho |
|---------|---------|
| Código EoMT | `/kaggle/working/eomt` |
| Dataset (upload) | `/kaggle/input/SEU_DATASET` |
| Modelo Qwen3 | `/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1` |
| Outputs / checkpoints | `/kaggle/working/eomt/outputs` |
| Arquivos persistentes | `/kaggle/working/` |

> Lembre-se: `/kaggle/input/` é somente leitura. Salve modelos e outputs em `/kaggle/working/`.